# Phase 1 - Data Cleaning
## ECON 5200 Project
### Card & Krueger (1994) Replication

This notebook prepares the raw Card and Krueger fast-food restaurant dataset for the replication analysis in Phase 2.

The main tasks in this notebook are:
- load the raw dataset
- assign variable names
- check missing values
- construct FTE employment variables
- save a cleaned dataset for later DID analysis

In [26]:
import pandas as pd
import numpy as np

In [27]:
url = "https://raw.githubusercontent.com/Jerrylu99/Econ-5200-Project/main/data/raw/public.dat"

col_names = [
    "sheet", "chain", "co_owned", "state",
    "southj", "centralj", "northj", "pa1", "pa2", "shore",
    "ncalls", "empft", "emppt", "nmgrs", "wage_st", "inctime", "firstinc",
    "bonus", "pctaff", "meals", "open", "hrsopen", "psoda", "pfry", "pentree",
    "nregs", "nregs11",
    "type2", "status2", "date2", "ncalls2", "empft2", "emppt2", "nmgrs2",
    "wage_st2", "inctime2", "firstin2", "special2", "meals2", "open2r",
    "hrsopen2", "psoda2", "pfry2", "pentree2", "nregs2", "nregs112"
]

df = pd.read_csv(
    url,
    sep=r"\s+",
    header=None,
    names=col_names,
    na_values="."
)

df.head()

,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,firstin2,special2,meals2,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1.0,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0.0,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0
2,506,2,1,0,0,0,0,1,0,0,...,0.25,NaN,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0


In [28]:
print("Dataset shape:", df.shape)
df.info()

Dataset shape: (410, 46)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410 entries, 0 to 409
Data columns (total 46 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sheet     410 non-null    int64  
 1   chain     410 non-null    int64  
 2   co_owned  410 non-null    int64  
 3   state     410 non-null    int64  
 4   southj    410 non-null    int64  
 5   centralj  410 non-null    int64  
 6   northj    410 non-null    int64  
 7   pa1       410 non-null    int64  
 8   pa2       410 non-null    int64  
 9   shore     410 non-null    int64  
 10  ncalls    410 non-null    int64  
 11  empft     404 non-null    float64
 12  emppt     406 non-null    float64
 13  nmgrs     404 non-null    float64
 14  wage_st   390 non-null    float64
 15  inctime   379 non-null    float64
 16  firstinc  367 non-null    float64
 17  bonus     410 non-null    int64  
 18  pctaff    366 non-null    float64
 19  meals     410 non-null    int64  
 20  open   

In [29]:
missing_table = df.isna().sum().sort_values(ascending=False)
missing_table[missing_table > 0]

,0
ncalls2,249
firstin2,80
inctime2,66
pctaff,44
firstinc,43
inctime,31
pfry2,28
nregs112,27
pentree2,24
psoda2,22


In [30]:
df["fte_before"] = df["empft"] + df["nmgrs"] + 0.5 * df["emppt"]
df["fte_after"] = df["empft2"] + df["nmgrs2"] + 0.5 * df["emppt2"]

df[["fte_before", "fte_after"]].describe()

,fte_before,fte_after
count,398.000000,396.000000
mean,20.998869,21.054293
std,9.749805,9.094453
min,5.000000,0.000000
25%,14.562500,14.500000
50%,19.500000,20.500000
75%,24.500000,26.500000
max,85.000000,60.500000


In [31]:
df["treat"] = df["state"]
df["treat"].value_counts()

,count
treat,
1,331
0,79


In [32]:
clean_df = df.copy()

clean_df = clean_df.dropna(subset=["fte_before", "fte_after"]).copy()

print("Cleaned dataset shape:", clean_df.shape)
clean_df.head()

Cleaned dataset shape: (384, 49)


,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112,fte_before,fte_after,treat
0,46,1,0,0,0,0,0,1,0,0,...,6.5,16.5,1.03,NaN,0.94,4.0,4.0,40.50,24.0,0
1,49,2,0,0,0,0,0,1,0,0,...,10.0,13.0,1.01,0.89,2.35,4.0,4.0,13.75,11.5,0
2,506,2,1,0,0,0,0,1,0,0,...,11.0,11.0,0.95,0.74,2.33,4.0,3.0,8.50,10.5,0
3,56,4,1,0,0,0,0,1,0,0,...,10.0,12.0,0.92,0.79,0.87,2.0,2.0,34.00,20.0,0
4,61,4,1,0,0,0,0,1,0,0,...,10.0,12.0,1.01,0.84,0.95,2.0,2.0,24.00,35.5,0


In [33]:
validation_table = clean_df.groupby("state")[["fte_before", "fte_after", "wage_st", "wage_st2"]].mean()
validation_table.index = ["PA", "NJ"]
validation_table

,fte_before,fte_after,wage_st,wage_st2
PA,23.380000,21.096667,4.630417,4.617246
NJ,20.430583,20.897249,4.608942,5.081200


In [34]:
import os

os.makedirs("data/clean", exist_ok=True)
clean_df.to_csv("data/clean/cleaned_data.csv", index=False)

print("Cleaned data saved to: data/clean/cleaned_data.csv")

Cleaned data saved to: data/clean/cleaned_data.csv


In [35]:
saved_df = pd.read_csv("data/clean/cleaned_data.csv")
print("Saved file shape:", saved_df.shape)
saved_df.head()

Saved file shape: (384, 49)


,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112,fte_before,fte_after,treat
0,46,1,0,0,0,0,0,1,0,0,...,6.5,16.5,1.03,NaN,0.94,4.0,4.0,40.50,24.0,0
1,49,2,0,0,0,0,0,1,0,0,...,10.0,13.0,1.01,0.89,2.35,4.0,4.0,13.75,11.5,0
2,506,2,1,0,0,0,0,1,0,0,...,11.0,11.0,0.95,0.74,2.33,4.0,3.0,8.50,10.5,0
3,56,4,1,0,0,0,0,1,0,0,...,10.0,12.0,0.92,0.79,0.87,2.0,2.0,34.00,20.0,0
4,61,4,1,0,0,0,0,1,0,0,...,10.0,12.0,1.01,0.84,0.95,2.0,2.0,24.00,35.5,0
